In [17]:
from datasets import load_dataset
import pandas as pd
import wikipediaapi

Caricamento dataset

In [19]:
dataset_fever = load_dataset("fever", "v1.0", trust_remote_code=True)
print(dataset_fever)

DatasetDict({
    train: Dataset({
        features: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id'],
        num_rows: 311431
    })
    labelled_dev: Dataset({
        features: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id'],
        num_rows: 37566
    })
    unlabelled_dev: Dataset({
        features: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id'],
        num_rows: 19998
    })
    unlabelled_test: Dataset({
        features: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id'],
        num_rows: 19998
    })
    paper_dev: Dataset({
        features: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id'],
        num_rows: 18999
    })
    paper_test: Dataset({
        features: [

# Studio dataset

Split dataset disponibili

In [23]:
print("Split disponibili: \n", list(dataset_fever.keys()))
print()
for nome_split, dati in dataset_fever.items():
    print(f"  {nome_split}: {len(dati):,} esempi")

Split disponibili: 
 ['train', 'labelled_dev', 'unlabelled_dev', 'unlabelled_test', 'paper_dev', 'paper_test']

  train: 311,431 esempi
  labelled_dev: 37,566 esempi
  unlabelled_dev: 19,998 esempi
  unlabelled_test: 19,998 esempi
  paper_dev: 18,999 esempi
  paper_test: 18,567 esempi


Struttura del dataset

In [3]:
print(dataset_fever["train"][0])

{'id': 75397, 'label': 'SUPPORTS', 'claim': 'Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.', 'evidence_annotation_id': 92206, 'evidence_id': 104971, 'evidence_wiki_url': 'Nikolaj_Coster-Waldau', 'evidence_sentence_id': 7}


In [25]:
print(dataset_fever["train"].features)

Schema del dataset (feature types):
{'id': Value(dtype='int32', id=None), 'label': Value(dtype='string', id=None), 'claim': Value(dtype='string', id=None), 'evidence_annotation_id': Value(dtype='int32', id=None), 'evidence_id': Value(dtype='int32', id=None), 'evidence_wiki_url': Value(dtype='string', id=None), 'evidence_sentence_id': Value(dtype='int32', id=None)}


Distribuzione delle label

In [29]:
df = pd.DataFrame(dataset_fever["train"])
print(df["label"].value_counts())

label
SUPPORTS           193756
REFUTES             70066
NOT ENOUGH INFO     47609
Name: count, dtype: int64


Lunghezza claim

In [31]:
df["lunghezza_claim"] = df["claim"].apply(lambda x: len(x.split()))
print(df["lunghezza_claim"].describe())

count    311431.000000
mean          7.588089
std           3.049780
min           2.000000
25%           5.000000
50%           7.000000
75%           9.000000
max          65.000000
Name: lunghezza_claim, dtype: float64


Valori mancanti e anomalie

In [45]:
print("Valori mancanti nel training set \n")
print(df_train.isnull().sum().to_string())

print("\nValori mancanti nel validation set \n")
print(df_val.isnull().sum().to_string())

Valori mancanti nel training set 

id                        0
label                     0
claim                     0
evidence_annotation_id    0
evidence_id               0
evidence_wiki_url         0
evidence_sentence_id      0
lunghezza_parole          0
lunghezza_caratteri       0

Valori mancanti nel validation set 

id                        0
label                     0
claim                     0
evidence_annotation_id    0
evidence_id               0
evidence_wiki_url         0
evidence_sentence_id      0


In [47]:
numero_duplicati = df_train["claim"].duplicated().sum()
print(f"Claim duplicate nel training set: {numero_duplicati:,}")

Claim duplicate nel training set: 175,886


# Modifiche dataset

Raggruppamento dataset per claim

In [50]:
df_per_claim = df_train.groupby("id").agg(
    testo_claim = ("claim", "first"),
    label = ("label", "first"),
    numero_evidenze = ("evidence_id", "count"),
    url_documenti = ("evidence_wiki_url", lambda x: list(x.dropna().unique())),
    sentence_ids = ("evidence_sentence_id", list)
).reset_index()

print(f"Righe totali nel dataset: {len(df_train):,}")
print(f"Claim uniche dopo raggruppamento: {len(df_per_claim):,}")

Righe totali nel dataset: 311,431
Claim uniche dopo raggruppamento: 145,449


In [52]:
print("Distribuzione numero evidenze per claim:")
print(df_per_claim["numero_evidenze"].describe().round(2).to_string())
print()
print("Conteggio per numero esatto di evidenze:")
print(df_per_claim["numero_evidenze"].value_counts().sort_index().to_string())

Distribuzione numero evidenze per claim:
count    145449.00
mean          2.14
std           3.66
min           1.00
25%           1.00
50%           1.00
75%           2.00
max         338.00

Conteggio per numero esatto di evidenze:
numero_evidenze
1      98547
2      22067
3       6355
4       4676
5       4115
6       2505
7       1437
8       1036
9        746
10       671
11       460
12       417
13       311
14       285
15       199
16       216
17       161
18       143
19       107
20       115
21        97
22        70
23        84
24        63
25        50
26        41
27        41
28        32
29        32
30        31
31        25
32        28
33        20
34        18
35        16
36        14
37        24
38        13
39        14
40         8
41         9
42        10
43         9
44         6
45         4
46        11
47         6
48         8
49         3
50         5
51         3
52         5
53         6
54         6
55         1
56         3
57         3
58      

In [55]:
print("Distribuzione label su claim uniche:")
conteggio_label = df_per_claim["label"].value_counts()
for label, n in conteggio_label.items():
    p = n / len(df_per_claim) * 100
    print(f"  {label:<20} {n:>7,}  ({p:4.1f}%)")

Distribuzione label su claim uniche:
  SUPPORTS              80,035  (55.0%)
  NOT ENOUGH INFO       35,639  (24.5%)
  REFUTES               29,775  (20.5%)


Analisi corpus wikipedia

In [57]:
pagine_uniche = df_train["evidence_wiki_url"].dropna().unique()
print(f"Pagine Wikipedia uniche citate nel train: {len(pagine_uniche):,}")

df_dev = pd.DataFrame(dataset_fever["labelled_dev"])
pagine_dev = df_dev["evidence_wiki_url"].dropna().unique()
print(f"Pagine Wikipedia uniche citate nel dev: {len(pagine_dev):,}")

overlap = set(pagine_uniche) & set(pagine_dev)
print(f"Pagine in comune tra train e dev: {len(overlap):,}")

Pagine Wikipedia uniche citate nel train: 12,550
Pagine Wikipedia uniche citate nel dev: 2,893
Pagine in comune tra train e dev: 909


Recupero testo evidence

In [60]:
def recupera_testo_pagina(titolo_pagina: str) -> str:
    wiki = wikipediaapi.Wikipedia(
        language='en',
        user_agent='FactChecker-ProgettoEsame/1.0'
    )
    pagina = wiki.page(titolo_pagina)
    if pagina.exists():
        return pagina.text
    return ""

def estrai_frase(testo_pagina: str, sentence_id: int) -> str:
    frasi = [f.strip() for f in testo_pagina.split('.') if f.strip()]
    if sentence_id < len(frasi):
        return frasi[sentence_id]
    return ""

esempio = df_per_claim[df_per_claim["label"] == "SUPPORTS"].iloc[0]
print(f"Claim:       {esempio['testo_claim']}")
print(f"Label:       {esempio['label']}")
print(f"Documento:   {esempio['url_documenti']}")
print(f"Sentence ID: {esempio['sentence_ids']}")
print()

if esempio['url_documenti']:
    titolo = esempio['url_documenti'][0]
    print(f"Recupero pagina Wikipedia: '{titolo}'...")
    testo = recupera_testo_pagina(titolo)
    if testo:
        print(f"Testo recuperato ({len(testo)} caratteri)")
        sid = esempio['sentence_ids'][0] if esempio['sentence_ids'] else 0
        frase = estrai_frase(testo, sid)
        print(f"\nFrase {sid}: {frase}")
    else:
        print("Pagina non trovata.")

Claim:       Chris Hemsworth appeared in A Perfect Getaway.
Label:       SUPPORTS
Documento:   ['Chris_Hemsworth']
Sentence ID: [2]

Recupero pagina Wikipedia: 'Chris_Hemsworth'...
Testo recuperato (17574 caratteri)

Frase 2: In the Marvel Cinematic Universe, Hemsworth starred as Thor in the 2011 film of the same name and reprised the role in several subsequent installments, which established him among the world's highest-paid actors


In [61]:
# STRATEGIA 2: Corpus ufficiale FEVER (per produzione)
# Verifica se è disponibile su HuggingFace
print("Tentativo caricamento corpus Wikipedia ufficiale FEVER...")
print()
try:
    corpus = load_dataset("fever", "wiki_pages", split="wikipedia_pages")
    print("✅ Corpus disponibile via HuggingFace!")
    print(f"   Numero pagine: {len(corpus):,}")
    print(f"   Primo documento: {corpus[0]}")
except Exception as e:
    print(f"⚠️  Non disponibile via HuggingFace ({e})")
    print()
    print("Scaricalo manualmente:")
    print("  1. Vai su https://fever.ai/dataset/fever.html")
    print("  2. Scarica 'wiki-pages.zip' (~3.1 GB)")
    print("  3. Estrailo in dati/grezzi/wiki_pages/")
    print("  4. Formato: JSONL, un doc per riga: {'id': ..., 'text': ..., 'lines': ...}")
    print()
    print("Per ora usiamo Wikipedia API per il prototipo.")

Tentativo caricamento corpus Wikipedia ufficiale FEVER...



Generating wikipedia_pages split: 100%|██████████| 5416537/5416537 [03:10<00:00, 28373.88 examples/s]


✅ Corpus disponibile via HuggingFace!
   Numero pagine: 5,416,537
   Primo documento: {'id': '', 'text': '', 'lines': ''}
